<a href="https://www.kaggle.com/code/insanejsk/query-rewriter-data-prep?scriptVersionId=327856120" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
"""
Query Rewriter - Synthetic Dataset Generation
===============================================
Reads clean queries from triples_train_200k.jsonl.
Applies random noise via rule-based corruption + LLM colloquial rephrasing.
Outputs (clean, noisy, flags) pairs for T5-small fine-tuning.

Noise flags (independently sampled per query):
  0: typos / misspellings - 0.5
  1: missing words - 0.5
  2: extra filler words - 0.5
  3: mixed case / no punctuation - 0.5
  4: colloquial rephrasing (LLM) - 0.2

Uses Qwen/Qwen2.5-1.5B-Instruct"

Output format (JSONL):
  {"clean": "...", "noisy": "...", "flags": [1, 0, 1, 1, 0]}
"""

import json
import os
import random
import re
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Config

CONFIG = {
    "input_train_path": "/kaggle/input/datasets/insanejsk/nse-dr/triples_train.jsonl",
    "output_train_path": "/kaggle/working/data/train.jsonl",
    "input_val_path": "/kaggle/input/datasets/insanejsk/nse-dr/triples_val.jsonl",
    "output_val_path": "/kaggle/working/data/val.jsonl",
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
    "noise_prob": [0.5, 0.5, 0.5, 0.5, 0.2],      # probability each flag independently fires
    "max_samples": 200000,
    "save_every": 1000,    # checkpoint save frequency
    "batch_size": 64
}

# TYPO TABLES

# common keyboard-adjacency typos
KEYBOARD_ADJACENCY = {
    'a': ['s', 'q', 'z'], 'b': ['v', 'n', 'g'], 'c': ['x', 'v', 'd'],
    'd': ['s', 'f', 'e', 'c'], 'e': ['w', 'r', 'd'], 'f': ['d', 'g', 'r'],
    'g': ['f', 'h', 't'], 'h': ['g', 'j', 'y'], 'i': ['u', 'o', 'k'],
    'j': ['h', 'k', 'u'], 'k': ['j', 'l', 'i'], 'l': ['k', 'o', 'p'],
    'm': ['n', 'k'], 'n': ['b', 'm', 'h'], 'o': ['i', 'p', 'l'],
    'p': ['o', 'l'], 'q': ['w', 'a'], 'r': ['e', 't', 'f'],
    's': ['a', 'd', 'w', 'z'], 't': ['r', 'y', 'g'], 'u': ['y', 'i', 'j'],
    'v': ['c', 'b', 'f'], 'w': ['q', 'e', 's'], 'x': ['z', 'c', 's'],
    'y': ['t', 'u', 'h'], 'z': ['a', 's', 'x'],
}

# common filler phrases
FILLERS = [
    "can you tell me", "i want to know", "please explain",
    "hey so", "like", "basically", "i was wondering",
    "could you help me understand", "so basically", "umm",
    "what exactly is", "lol", "pls", "thanks",
]

# NOISE FUNCTIONS

def apply_typos(text):
    """Randomly corrupt ~15% of characters with keyboard-adjacent swaps,
    duplications, or deletions."""
    chars  = list(text)
    result = []
    for ch in chars:
        r = random.random()
        lower = ch.lower()
        if r < 0.05 and lower in KEYBOARD_ADJACENCY:
            # replace with adjacent key
            result.append(random.choice(KEYBOARD_ADJACENCY[lower]))
        elif r < 0.08:
            # duplicate character
            result.append(ch)
            result.append(ch)
        elif r < 0.11:
            continue
        else:
            result.append(ch)
    return "".join(result)


def apply_missing_words(text):
    """Randomly drop 20-40% of words, keeping at least 2."""
    words = text.split()
    if len(words) <= 2:
        return text
    keep_prob = random.uniform(0.6, 0.8)
    kept = [w for w in words if random.random() < keep_prob]
    if len(kept) < 2:
        kept = random.sample(words, 2)
    return " ".join(kept)


def apply_filler(text):
    """Add filler phrase at start or end."""
    filler = random.choice(FILLERS)
    if random.random() < 0.5:
        return f"{filler} {text}"
    else:
        return f"{text} {filler}"

def apply_case_punct(text):
    """Random case + remove punctuation."""
    choice = random.random()
    if choice < 0.33:
        text = text.lower()
    elif choice < 0.66:
        text = text.upper()
    else:
        text = text.title()
    # randomly remove punctuation
    if random.random() < 0.7:
        text = re.sub(r"[^\w\s]", "", text)
    return text


def apply_rule_based_noise(text, flags):
    """Apply flags 0-3 in sequence."""
    if flags[0]:
        text = apply_typos(text)
    if flags[1]:
        text = apply_missing_words(text)
    if flags[2]:
        text = apply_filler(text)
    if flags[3]:
        text = apply_case_punct(text)
    return text

# LLM COLLOQUIAL REPHRASING

SYSTEM_PROMPT = """
You generate synthetic training data.

For every input query return exactly one JSON object.

Format:

{"rewrite":"your rewritten query"}

Rules:
- Keep meaning unchanged.
- Make it casual, colloquial and slightly messy.
- No explanations.
- No markdown.
- No emojis.
- Return JSON only.
"""

print("Loading Qwen...")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"]
)

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("Qwen loaded")

def batch_colloquial_rephrase(texts):
    prompts = [
            f"""
    Return ONLY valid JSON.
    
    {{"rewrite":"..."}}
    
    Query:
    {text}
    """
        for text in texts
    ]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=True,
            temperature=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True,
    )

    results = []

    for text in decoded:

        match = re.search(
            r'\{.*\}',
            text,
            flags=re.DOTALL
        )

        if match:

            try:

                obj = json.loads(
                    match.group(0)
                )

                results.append(
                    obj["rewrite"].strip()
                )

                continue

            except Exception:
                pass

        results.append(text.strip())

    return results

def generate(val=0):
    "load queries and deduplicate"
    os.makedirs("/kaggle/working/data/", exist_ok=True)
    if val == 0:
        CONFIG["input_path"] = CONFIG["input_train_path"]
        CONFIG["output_path"] = CONFIG["output_train_path"]
    else:
        CONFIG["input_path"] = CONFIG["input_val_path"]
        CONFIG["output_path"] = CONFIG["output_val_path"]
    print(f"Loading queries from {CONFIG['input_path']}...")
    queries = set()
    with open(CONFIG["input_path"]) as f:
        for line in f:
            obj = json.loads(line)
            queries.add(obj["query"])
    queries = list(queries)[:CONFIG["max_samples"]]
    print(f"Unique queries: {len(queries):,}")
    
    # load existing progress if resuming
    done = set()
    out_file = CONFIG["output_path"]
    if os.path.exists(out_file):
        with open(out_file) as f:
            for line in f:
                obj = json.loads(line)
                done.add(obj["clean"])
        print(f"Resuming — {len(done):,} already done")
    
    queries = [q for q in queries if q not in done]
    print(f"Remaining: {len(queries):,}")
    stats = {"qwen": 0, "skipped": 0, "no_noise": 0}
    records = []
    pending_colloquial = []

    print("Applying rule-based noise...")
    for query in tqdm(
        queries,
        desc="Rule-based noise"
    ):
        flags = [
            1 if random.random() < CONFIG["noise_prob"][i]
            else 0
            for i in range(5)
        ]
        noisy = apply_rule_based_noise(
            query,
            flags
        )
        if sum(flags) == 0:
            stats["no_noise"] += 1
        if flags[4]:
            pending_colloquial.append({
                "clean": query,
                "noisy": noisy,
                "flags": flags,
            })
        else:
            records.append({
                "clean": query,
                "noisy": noisy,
                "flags": flags,
            })
    print(
        f"Collected {len(pending_colloquial):,} "
        f"Qwen rewrites"
    )

    for start_idx in tqdm(
        range(
            0,
            len(pending_colloquial),
            CONFIG["batch_size"]
        ),
        desc="Qwen batches"
    ):
        batch = pending_colloquial[
            start_idx:
            start_idx + CONFIG["batch_size"]
        ]
        rewrites = batch_colloquial_rephrase(
            [item["noisy"] for item in batch]
        )
        for item, rewrite in zip(
            batch,
            rewrites
        ):

            item["noisy"] = rewrite

            records.append(item)

            stats["qwen"] += 1

    print("Writing output...")

    with open(
        out_file,
        "w",
        encoding="utf-8"
    ) as f:

        for record in tqdm(
            records,
            desc="Saving"
        ):

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )

    total = len(records)

    print(f"\n{'='*50}")
    print("Generation Complete")
    print(f"Total pairs    : {total:,}")
    print(f"Qwen rewrites : {stats['qwen']:,}")
    print(
        f"No-noise pairs : "
        f"{stats['no_noise']:,}"
    )
    print(f"Saved to       : {out_file}")
    print(f"{'='*50}")

    print("\nSample pairs:")

    for sample in records[:5]:

        print(
            f"\nflags={sample['flags']}"
        )

        print(
            f"clean : {sample['clean']}"
        )

        print(
            f"noisy : {sample['noisy']}"
        )

# if __name__ == "__main__":
#     generate(1)
#     generate(0)

Loading Qwen...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded


In [3]:
# Since word drops wasn't capped, some examples got irrecoverable
# clean : can i drink milk gout
# noisy : thanks can idrink
# We'll be editing the generated dataset to regenerate all those samples with word drop flag True
# using the SAME flags but with length-aware drop cap
# Overwrite in place

CONFIG["input_train_path"] = "/kaggle/input/datasets/insanejsk/queryrewriter-synthetic-ms-marco/data/train.jsonl"
CONFIG["input_val_path"] = "/kaggle/input/datasets/insanejsk/queryrewriter-synthetic-ms-marco/data/val.jsonl"
CONFIG["output_train_path"]= "/kaggle/working/data/train.jsonl"
CONFIG["output_val_path"] = "/kaggle/working/data/val.jsonl"

# New missing_word_function
def apply_missing_words(text):
    """Length-aware drop: max 1 word for <=5, max 2 for <=10, 25% cap beyond."""
    words = text.split()
    n = len(words)
    if n <= 2:
        return text  # never drop from 2-word queries
    if n <= 5:
        max_drop = 1
    elif n <= 10:
        max_drop = 2
    else:
        max_drop = max(2, int(n * 0.25))
    n_drop  = random.randint(1, max_drop)
    indices = sorted(random.sample(range(n), n_drop), reverse=True)
    for i in indices:
        words.pop(i)
    return " ".join(words) if words else text

def apply_flags(clean, flags):
    noisy = clean
    if flags[0]: noisy = apply_typos(noisy)
    if flags[1]: noisy = apply_missing_words(noisy)   # fixed function name
    if flags[2]: noisy = apply_filler(noisy)
    if flags[3]: noisy = apply_case_punct(noisy)      # correct function name
    return noisy

def fix_file(in_path, out_path):
    os.makedirs("/kaggle/working/data/", exist_ok=True)
    pairs = []
    with open(in_path) as f:
        for line in f:
            pairs.append(json.loads(line))

    to_fix = [i for i, p in enumerate(pairs) if p["flags"][1] == 1]
    print(f"{in_path}: {len(pairs)} total, {len(to_fix)} to fix")

    # Split into flag-4 and non-flag-4
    to_fix_llm   = [i for i in to_fix if pairs[i]["flags"][4] == 1]
    to_fix_rules = [i for i in to_fix if pairs[i]["flags"][4] == 0]

    # Fix rule-based only pairs
    for i in tqdm(to_fix_rules, desc="Fixing rule-based"):
        pairs[i]["noisy"] = apply_flags(pairs[i]["clean"], pairs[i]["flags"])

    # Fix flag-4 pairs in batches
    for start in tqdm(range(0, len(to_fix_llm), CONFIG["batch_size"]), desc="Fixing Qwen"):
        batch_idx = to_fix_llm[start : start + CONFIG["batch_size"]]
        # Re-apply rule-based noise first, then pass to Qwen
        rule_noisy = [apply_flags(pairs[i]["clean"], pairs[i]["flags"]) for i in batch_idx]
        rewrites   = batch_colloquial_rephrase(rule_noisy)
        for i, rewrite in zip(batch_idx, rewrites):
            pairs[i]["noisy"] = rewrite

    with open(out_path, "w") as f:
        for p in pairs:
            f.write(json.dumps(p) + "\n")
    print(f"  Saved to {out_path}")

fix_file(CONFIG["input_train_path"], CONFIG["output_train_path"])
fix_file(CONFIG["input_val_path"],   CONFIG["output_val_path"])
print("Done.")

/kaggle/input/datasets/insanejsk/queryrewriter-synthetic-ms-marco/data/train.jsonl: 200000 total, 100089 to fix


Fixing Qwen: 100%|██████████| 315/315 [12:22<00:00,  2.36s/it]


  Saved to /kaggle/working/data/train.jsonl
/kaggle/input/datasets/insanejsk/queryrewriter-synthetic-ms-marco/data/val.jsonl: 5000 total, 2544 to fix


Fixing Qwen: 100%|██████████| 8/8 [00:18<00:00,  2.32s/it]

  Saved to /kaggle/working/data/val.jsonl
Done.
